# 01 — Figure: interface geometry (P_PCA và gauge R)

Notebook canonical cho hình bốn panel mô tả teacher interface: teacher gốc, student
pretrained, target sau khi giảm chiều + chọn toạ độ, và phân bố cosine mà `R` thật sự
tối ưu.

Ba bảo đảm để hình là **bằng chứng** chứ không phải minh hoạ:

- **Mọi phép fit chạy ở `d_S` chiều gốc.** `P_PCA` và `R` được fit bằng đúng các hàm
  `src/teacher_projection.py` mà `distiller.py` gọi lúc train. 2D chỉ là cách nhìn,
  không có đại lượng nào được tính sau khi chiếu xuống 2D.
- **Một khung nhìn duy nhất cho panel (c).** `W_Z` fit một lần trên student pretrained
  rồi dùng chung cho `Y`, `YQ`, `YR*` và `Z_0`. PCA tự nó được phép xoay trục, nên fit
  PCA riêng cho từng đám mây sẽ tạo ra "alignment" từ chính visualization.
- **Fit trên toàn bộ fit set, subsample chỉ để vẽ.** `R*` fit trên cả 14,760 câu như
  run thật, sau đó mới lấy `N_PLOT` điểm để scatter.

Cell 5 tự đối chiếu số nó fit được với log của run headline
(`runs/pca_384__selected/seed_42/train.log`: retains 92.8%, cosine −0.013 → +0.654,
participation ratio 1.37). Lệch quá ngưỡng là fail — nghĩa là hình đang mô tả một
thí nghiệm khác với bảng trong bài.

**Lưu ý cho caption**: hình vẽ `R` tại thời điểm khởi tạo. Recipe mặc định
(`gauge_refit_every = 1`) refit `R` mỗi epoch.

In [ ]:
# 1. Cấu hình hình
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"

PAIRS = {
    "qwen3_0.6b_to_minilm_h384": {
        "teacher": "Qwen/Qwen3-Embedding-0.6B",
        "student": "nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base",
        "teacher_pooling": "last_token",
        "student_pooling": "cls",
        "min_vram_gib": 12,
    },
    "bge_m3_to_minilm_h768": {
        "teacher": "BAAI/bge-m3",
        "student": "nreimers/MiniLMv2-L6-H768-distilled-from-BERT-Base",
        "teacher_pooling": "cls",
        "student_pooling": "cls",
        "min_vram_gib": 12,
    },
    "qwen3_4b_to_bert_base": {
        "teacher": "Qwen/Qwen3-Embedding-4B",
        "student": "google-bert/bert-base-uncased",
        "teacher_pooling": "last_token",
        "student_pooling": "cls",
        "min_vram_gib": 24,
    },
}
# Cặp headline của bài: cùng cặp mà runs/pca_384__selected đã train, nên số fit được
# ở cell 5 phải trùng log của run đó.
PAIR = "qwen3_0.6b_to_minilm_h384"

# Câu để encode. Corpus train là mặc định vì đó là tập mà P_PCA và R được fit trên
# đúng lúc train; một test split cũng chạy được và trả lời câu hỏi khác ("interface
# nhìn thế nào trên dữ liệu chưa từng thấy"), nhưng khi đó self-check ở cell 5 tắt.
SENTENCE_SOURCES = {
    "talas_15k": Path("data/train_set/merged_3_data_5k_each.csv"),
    "100k": Path("data/train_set/train_100k.csv"),
    "msmarco_15k_disjoint": Path("data/train_set/corpus_disjoint_msmarco_15k.csv"),
    "stsb_test": Path("data/test_set/stsb_test.csv"),
    "sick_test": Path("data/test_set/sick_test.csv"),
}
SOURCE = "talas_15k"

MAX_LENGTH = 256
# 0 = dùng toàn bộ source. R* phải được fit trên cả fit set rồi mới subsample để vẽ:
# fit trên đúng những điểm sẽ vẽ là một transformation riêng cho hình đó.
N_FIT = 0
# Số điểm scatter. Đủ để thấy hình dạng, không đủ để panel thành một vũng mực.
N_PLOT = 800
PLOT_SEED = 0
# Trùng gauge_random_seed mặc định, nên Q ở đây là đúng ma trận mà arm
# --gauge_rotation random dùng.
HAAR_SEED = 0
# Số câu neo được đánh dấu ở cả bốn panel để đọc correspondence mà không cần colorbar.
N_ANCHORS = 8

# Khung nhìn 2D fit trên Z_0 chưa center. Các vector nằm trên mặt cầu và loss là
# cosine, nên mean direction là một phần hình học thật của student chứ không phải
# nuisance; center trước khi fit sẽ ném đúng cái direction mà R* xoay ra khỏi hình
# (participation ratio 1.37: phép xoay gần như rank-one). Đổi thành True để xem
# phần hình học còn lại sau khi bỏ mean.
FRAME_CENTER = False

# Log của run headline. Self-check chỉ chạy khi cấu hình khớp run đó.
REFERENCE_RUN = "runs/pca_384__selected/seed_42/train.log"
REFERENCE_STATS = {
    "explained_energy": 0.928,
    "cos_before": -0.013,
    "cos_after": 0.654,
    "participation_ratio": 1.37,
    "samples": 14760,
}
REFERENCE_TOLERANCE = 0.02

FIGURE_STEM = "interface_geometry"
RUN_STAMP = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%Y%m%d-%H%M%S")

PAIR_CONFIG = PAIRS[PAIR]
TEACHER_MODEL = PAIR_CONFIG["teacher"]
STUDENT_MODEL = PAIR_CONFIG["student"]
TEACHER_POOLING = PAIR_CONFIG["teacher_pooling"]
STUDENT_POOLING = PAIR_CONFIG["student_pooling"]
SOURCE_PATH = SENTENCE_SOURCES[SOURCE]
IS_REFERENCE_CONFIG = (
    PAIR == "qwen3_0.6b_to_minilm_h384" and SOURCE == "talas_15k" and N_FIT == 0
)

print(f"Pair: {PAIR}")
print(f"Sentences: {SOURCE} ({SOURCE_PATH})")
print(f"Self-check với {REFERENCE_RUN}: {'bật' if IS_REFERENCE_CONFIG else 'tắt'}")

In [ ]:
# 2. Dùng repo hiện tại hoặc clone trên Colab; cài dependencies.
import subprocess
import sys

cwd = Path.cwd().resolve()
LOCAL_REPO = (cwd / "main.py").is_file() and (cwd / "distiller.py").is_file()
if LOCAL_REPO:
    PROJECT_DIR = cwd
else:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if PROJECT_DIR.exists():
        assert (PROJECT_DIR / "main.py").is_file(), f"Repo không hợp lệ: {PROJECT_DIR}"
    else:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

# Chỉ pull ở bản clone: checkout local là nơi bài đang được viết và thường có thay
# đổi chưa commit, --ff-only sẽ hoặc fail hoặc kéo mất trạng thái đang làm dở.
if not LOCAL_REPO:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
        check=True,
    )
head = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print(f"Project: {PROJECT_DIR} @ {head}")

In [ ]:
# 3. Output, device và câu
import os

import pandas as pd
import torch

sys.path.insert(0, str(PROJECT_DIR))

FIGURE_DIR = PROJECT_DIR / "docs" / "latex_iclr" / "figures"
EMBEDDING_CACHE_DIR = PROJECT_DIR / "runs" / "figure_cache"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_CACHE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"cuda:0: {props.name} ({props.total_memory / 2**30:.1f} GiB)")
    if props.total_memory / 2**30 < PAIR_CONFIG["min_vram_gib"]:
        print(f"[WARN] Nên có >= {PAIR_CONFIG['min_vram_gib']} GiB cho teacher này.")
else:
    print("[WARN] Không có GPU: một lượt forward teacher qua toàn bộ corpus trên CPU "
          "mất hàng giờ. Cell 4 cache kết quả nên chỉ phải trả giá một lần.")

source_path = PROJECT_DIR / SOURCE_PATH
if not source_path.is_file():
    raise FileNotFoundError(f"Thiếu dữ liệu: {source_path}")
frame = pd.read_csv(source_path)

# Cùng quy tắc cột với Distiller._cache_texts: teacher chỉ encode text thứ nhất của
# mỗi dòng, nên hình đọc đúng những chuỗi mà cache lúc train đã đọc.
for column in ("premise", "sentence1", "text"):
    if column in frame.columns:
        TEXT_COLUMN = column
        break
else:
    raise ValueError(f"Không tìm thấy cột text trong {source_path}: {list(frame.columns)}")

TEXTS = frame[TEXT_COLUMN].astype(str).tolist()
if N_FIT:
    TEXTS = TEXTS[:N_FIT]
print(f"{len(TEXTS)} câu từ cột '{TEXT_COLUMN}'")
print(f"Figures: {FIGURE_DIR}")

In [ ]:
# 4. Encode teacher và student pretrained trên cùng tập câu (có cache)
import hashlib

import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

from src.structural_audit import encode_texts

ENCODE_BATCH_SIZE = 64 if DEVICE == "cuda" else 8


def _load_model(name, dtype=None):
    """AutoModel.from_pretrained, chịu được cả tên keyword cũ lẫn mới của dtype.

    transformers >= 5 đổi ``torch_dtype`` thành ``dtype``; distiller.py xử lý chuyện
    này ở chỗ khác, notebook thì phải tự chịu vì Colab pin phiên bản riêng.
    """
    kwargs = {"trust_remote_code": True}
    if dtype is None:
        return AutoModel.from_pretrained(name, **kwargs)
    try:
        return AutoModel.from_pretrained(name, dtype=dtype, **kwargs)
    except TypeError:
        return AutoModel.from_pretrained(name, torch_dtype=dtype, **kwargs)


identity = "|".join([
    TEACHER_MODEL, STUDENT_MODEL, TEACHER_POOLING, STUDENT_POOLING,
    str(MAX_LENGTH), str(len(TEXTS)),
    hashlib.sha256("\n".join(TEXTS).encode("utf-8")).hexdigest()[:16],
])
cache_path = EMBEDDING_CACHE_DIR / f"{PAIR}__{SOURCE}__{hashlib.sha256(identity.encode()).hexdigest()[:12]}.pt"

if cache_path.is_file():
    payload = torch.load(cache_path, map_location="cpu", weights_only=False)
    assert payload["identity"] == identity, f"Cache không khớp cấu hình: {cache_path}"
    print(f"Đọc lại embeddings đã cache: {cache_path}")
else:
    # Teacher: đúng pooling và đúng normalize của cache lúc train (normalize_cache=True),
    # nên T ở đây là chính ma trận mà P_PCA được fit trên.
    print(f"Encoding teacher {TEACHER_MODEL} ...")
    teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL, trust_remote_code=True, use_fast=True)
    teacher_model = _load_model(TEACHER_MODEL, torch.bfloat16 if DEVICE == "cuda" else None)
    teacher_model.eval().to(DEVICE)
    with torch.inference_mode():
        teacher_raw = encode_texts(
            teacher_model, teacher_tokenizer, TEXTS, device=DEVICE,
            pooling=TEACHER_POOLING, batch_size=ENCODE_BATCH_SIZE,
            max_length=MAX_LENGTH, amp=DEVICE == "cuda", progress=True,
        )["final"]
    del teacher_model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    # Student *trước* distill, cùng pooling và cùng normalize với
    # Distiller._student_initial_embeddings -- đó là ma trận Procrustes fit vào.
    print(f"Encoding student {STUDENT_MODEL} (pretrained, chưa distill) ...")
    student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL, use_fast=True)
    student_model = _load_model(STUDENT_MODEL)
    student_model.eval().to(DEVICE)
    with torch.inference_mode():
        student_raw = encode_texts(
            student_model, student_tokenizer, TEXTS, device=DEVICE,
            pooling=STUDENT_POOLING, batch_size=ENCODE_BATCH_SIZE,
            max_length=MAX_LENGTH, amp=DEVICE == "cuda", progress=True,
        )["final"]
    del student_model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    payload = {
        "teacher": F.normalize(teacher_raw.float(), dim=-1).clone(),
        "student": F.normalize(student_raw.float(), dim=-1).clone(),
        "identity": identity,
        "pair": PAIR, "source": SOURCE, "texts": len(TEXTS),
        "teacher_model": TEACHER_MODEL, "student_model": STUDENT_MODEL,
        "teacher_pooling": TEACHER_POOLING, "student_pooling": STUDENT_POOLING,
        "max_length": MAX_LENGTH, "stamp": RUN_STAMP,
    }
    torch.save(payload, cache_path)
    print(f"Đã ghi {cache_path}")

T = payload["teacher"]
Z0 = payload["student"]
D_T, D_S = T.shape[1], Z0.shape[1]
print(f"T: {tuple(T.shape)}   Z0: {tuple(Z0.shape)}")

In [ ]:
# 5. Fit P_PCA và gauge R ở d_S chiều gốc, bằng đúng các hàm distiller.py gọi
import numpy as np

from src.teacher_projection import (
    fit_gauge_alignment,
    fit_pca_projection,
    project_teacher_embeddings,
    random_orthogonal,
    retained_energy,
)

RANK = min(D_S, D_T)
# center=True / subtract_mean=False là cấu hình của bài: mean được bỏ khi *chọn*
# hướng, nhưng map được apply đúng dạng tuyến tính P_T (geoode_config.py:106-109).
P_PCA, TEACHER_MEAN = fit_pca_projection(T, out_dim=RANK, center=True)
if RANK < D_S:
    P_PCA = F.pad(P_PCA, (0, D_S - RANK)).contiguous()
Y = project_teacher_embeddings(T, P_PCA, mean=TEACHER_MEAN, subtract_mean=False)

R_STAR, GAUGE_STATS = fit_gauge_alignment(Y, Z0)
Q_HAAR = random_orthogonal(D_S, seed=HAAR_SEED)
Y_ROT = F.normalize(Y @ R_STAR, dim=-1)
Y_HAAR = F.normalize(Y @ Q_HAAR, dim=-1)
ENERGY = retained_energy(T, P_PCA)

# Cosine từng câu, đo ở d_S chiều -- đại lượng R* thật sự tối ưu, không dính
# projection nào. Panel (d) vẽ đúng ba vector này.
COSINES = {
    "PCA only": (Y * Z0).sum(-1).numpy(),
    "PCA + Haar Q": (Y_HAAR * Z0).sum(-1).numpy(),
    "PCA + Procrustes R": (Y_ROT * Z0).sum(-1).numpy(),
}

print(f"rank-{RANK} PCA {D_T} -> {D_S}: giữ {ENERGY:.1%} năng lượng "
      f"(subspace ngẫu nhiên cùng hạng giữ ~{RANK / D_T:.1%})")
print(f"gauge trên {GAUGE_STATS['samples']} câu: cosine "
      f"{GAUGE_STATS['cos_before']:+.3f} -> {GAUGE_STATS['cos_after']:+.3f}")
print(f"participation ratio {GAUGE_STATS['participation_ratio']:.2f} / {D_S} "
      f"(top singular share {GAUGE_STATS['top_singular_share']:.3f})")
print(f"Haar control: cosine {COSINES['PCA + Haar Q'].mean():+.3f}")

if IS_REFERENCE_CONFIG:
    # Hình phải mô tả đúng thí nghiệm trong bảng. Fit ở đây độc lập với run train,
    # nên nếu nó không tái tạo được số trong log thì một trong hai chỗ sai -- và
    # thà biết ngay còn hơn để hình đẹp mà nói chuyện khác.
    observed = {
        "explained_energy": float(ENERGY),
        "cos_before": GAUGE_STATS["cos_before"],
        "cos_after": GAUGE_STATS["cos_after"],
        "participation_ratio": GAUGE_STATS["participation_ratio"],
        "samples": GAUGE_STATS["samples"],
    }
    drift = {
        key: (observed[key], REFERENCE_STATS[key])
        for key in REFERENCE_STATS
        if abs(observed[key] - REFERENCE_STATS[key]) > (
            0 if key == "samples" else REFERENCE_TOLERANCE
        )
    }
    assert not drift, f"Lệch so với {REFERENCE_RUN}: {drift}"
    print(f"Self-check: khớp {REFERENCE_RUN} trong sai số {REFERENCE_TOLERANCE}")

In [ ]:
# 6. Khung nhìn 2D. Fit một lần, dùng chung -- fit riêng từng đám mây là tự tạo alignment.
def viewing_frame(matrix, center=FRAME_CENTER):
    """Hai hướng chính của ``matrix`` cùng gốc toạ độ đi kèm, dấu đã cố định.

    Trả ``(mu, W)`` để mọi đám mây được nhìn qua đúng một affine map
    ``x -> (x - mu) W``. Dấu của mỗi cột do LAPACK chọn tuỳ ý, nên hình sẽ lật
    ngẫu nhiên giữa các lần chạy nếu không ép: quy ước ở đây là thành phần có trị
    tuyệt đối lớn nhất của mỗi cột phải dương.
    """
    mu = matrix.mean(dim=0) if center else torch.zeros(matrix.shape[1])
    _, _, vh = torch.linalg.svd(matrix - mu, full_matrices=False)
    basis = vh[:2].transpose(0, 1).contiguous()
    signs = torch.sign(basis[basis.abs().argmax(dim=0), torch.arange(2)])
    return mu, basis * torch.where(signs == 0, torch.ones_like(signs), signs)


def project(matrix, frame):
    mu, basis = frame
    return ((matrix - mu) @ basis).numpy()


TEACHER_FRAME = viewing_frame(T)      # panel (a): không gian teacher, d_T chiều
STUDENT_FRAME = viewing_frame(Z0)     # panel (b) và (c): khung dùng chung

rng = np.random.default_rng(PLOT_SEED)
PLOT_INDEX = np.sort(rng.choice(len(TEXTS), size=min(N_PLOT, len(TEXTS)), replace=False))
ANCHOR_INDEX = PLOT_INDEX[np.linspace(0, len(PLOT_INDEX) - 1, N_ANCHORS).round().astype(int)]

VIEW = {
    "T": project(T, TEACHER_FRAME),
    "Z0": project(Z0, STUDENT_FRAME),
    "Y": project(Y, STUDENT_FRAME),
    "Y_haar": project(Y_HAAR, STUDENT_FRAME),
    "Y_rot": project(Y_ROT, STUDENT_FRAME),
}

# Mỗi hàng là vector đơn vị ở d chiều, nên bóng 2D của nó nằm trong đĩa bán kính 1.
# Trục của panel thường nhỏ hơn thế rất nhiều; con số này được in lên hình để không
# ai đọc kích thước đám mây như thể nó là toàn bộ vector.
SPAN = {key: float(np.abs(value).max()) for key, value in VIEW.items()}
for key, value in SPAN.items():
    print(f"{key:6s}: |toạ độ| lớn nhất = {value:.3f} (bán kính mặt cầu = 1)")

In [ ]:
# 7. Render bốn panel
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

INK, MUTED, PANEL = "#202124", "#69707D", "#FAFAF8"
TEACHER_C, HAAR_C, GATE_C, STUDENT_C = "#B45F06", "#A9B0BC", "#5B43B4", "#596579"
SHOW_HAAR = True

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8,
    "figure.dpi": 140, "savefig.dpi": 300,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})


def clean(ax, title, subtitle):
    ax.set_facecolor(PANEL)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_color("#DDDDDD")
    ax.set_title(title, fontsize=8.5, color=INK, pad=9, loc="left")
    ax.text(0.0, 1.015, subtitle, transform=ax.transAxes, fontsize=7,
            color=MUTED, va="bottom")


def square_limits(ax, clouds, pad=0.12):
    """Cùng một tỉ lệ cho mọi đám mây trong panel, gốc toạ độ giữ nguyên vị trí."""
    points = np.vstack(clouds)
    half = np.abs(points).max() * (1 + pad)
    ax.set_xlim(-half, half)
    ax.set_ylim(-half, half)
    ax.set_aspect("equal")
    return half


def scale_note(ax, half):
    ax.text(0.02, 0.03, f"trục = {half:.2f} của bán kính mặt cầu 1.00",
            transform=ax.transAxes, fontsize=6, color=MUTED)


figure = plt.figure(figsize=(11.6, 3.0))
grid = figure.add_gridspec(1, 4, width_ratios=[1, 1, 1.3, 1.15], wspace=0.16)
ax_a, ax_b, ax_c, ax_d = (figure.add_subplot(grid[0, index]) for index in range(4))

plot_slice = PLOT_INDEX
anchor_kw = dict(s=26, facecolors="none", edgecolors=INK, linewidths=0.7, zorder=5)

# (a) teacher gốc, nhìn qua hai hướng chính của chính nó
ax_a.scatter(*VIEW["T"][plot_slice].T, s=4, c=TEACHER_C, alpha=0.35, linewidths=0)
ax_a.scatter(*VIEW["T"][ANCHOR_INDEX].T, **anchor_kw)
clean(ax_a, "(a) Teacher $T$", f"PCA 2D của chính nó · $d_T={D_T}$")
scale_note(ax_a, square_limits(ax_a, [VIEW["T"][plot_slice]]))

# (b) student pretrained, khung nhìn được giữ lại cho panel (c)
ax_b.scatter(*VIEW["Z0"][plot_slice].T, s=4, c=STUDENT_C, alpha=0.35, linewidths=0)
ax_b.scatter(*VIEW["Z0"][ANCHOR_INDEX].T, **anchor_kw)
clean(ax_b, "(b) Student $Z_0$ (chưa distill)", f"PCA 2D $W_Z$ · $d_S={D_S}$")
scale_note(ax_b, square_limits(ax_b, [VIEW["Z0"][plot_slice]]))

# (c) mọi thứ trong đúng khung W_Z của (b): so sánh được vì không đám mây nào
# được cho phép tự chọn trục của mình.
clouds = [VIEW["Z0"][plot_slice], VIEW["Y"][plot_slice], VIEW["Y_rot"][plot_slice]]
ax_c.scatter(*VIEW["Z0"][plot_slice].T, s=9, c=STUDENT_C, alpha=0.30, marker="x", linewidths=0.5)
ax_c.scatter(*VIEW["Y"][plot_slice].T, s=5, facecolors="none", edgecolors=TEACHER_C,
             alpha=0.45, linewidths=0.4)
if SHOW_HAAR:
    clouds.append(VIEW["Y_haar"][plot_slice])
    ax_c.scatter(*VIEW["Y_haar"][plot_slice].T, s=4, c=HAAR_C, alpha=0.40, linewidths=0)
ax_c.scatter(*VIEW["Y_rot"][plot_slice].T, s=5, c=GATE_C, alpha=0.60, linewidths=0)
for index in ANCHOR_INDEX:
    ax_c.annotate(
        "", xy=VIEW["Y_rot"][index], xytext=VIEW["Y"][index],
        arrowprops=dict(arrowstyle="->", color=INK, lw=0.55, alpha=0.75,
                        shrinkA=1.5, shrinkB=1.5),
    )
clean(ax_c, "(c) Reduction + chọn toạ độ", f"cùng khung $W_Z$ · fit ở {D_S}D trên {GAUGE_STATS['samples']:,} câu")
scale_note(ax_c, square_limits(ax_c, clouds))
handles = [
    Line2D([], [], marker="x", color=STUDENT_C, lw=0, markersize=4, label="$Z_0$ student"),
    Line2D([], [], marker="o", color=TEACHER_C, lw=0, markersize=3.5, markerfacecolor="none",
           label="$Y=\\mathrm{norm}(TP)$"),
    Line2D([], [], marker="o", color=GATE_C, lw=0, markersize=3.5, label="$YR^{\\star}$"),
]
if SHOW_HAAR:
    handles.insert(2, Line2D([], [], marker="o", color=HAAR_C, lw=0, markersize=3.5,
                             label="$YQ_{\\mathrm{Haar}}$"))
ax_c.legend(handles=handles, loc="upper right", fontsize=6, frameon=False,
            handletextpad=0.3, borderaxespad=0.2, labelspacing=0.25)

# (d) đại lượng R* thật sự tối ưu, đo ở d_S chiều trên toàn bộ fit set. Panel này
# là chỗ duy nhất không có bước chiếu nào, nên nó là bằng chứng cho (c).
bins = np.linspace(-0.6, 1.0, 61)
for label, color in (("PCA only", TEACHER_C), ("PCA + Haar Q", HAAR_C),
                     ("PCA + Procrustes R", GATE_C)):
    values = COSINES[label]
    ax_d.hist(values, bins=bins, color=color, alpha=0.55, linewidth=0, label=label)
    ax_d.axvline(values.mean(), color=color, lw=1.0, ls="--", alpha=0.9)
ax_d.set_facecolor(PANEL)
for spine in ax_d.spines.values():
    spine.set_color("#DDDDDD")
ax_d.set_yticks([])
ax_d.tick_params(axis="x", labelsize=6.5, colors=MUTED, length=2)
ax_d.set_xlabel(r"$\cos(z_i,\,\tau_i)$ ở $d_S$ chiều", fontsize=7, color=INK)
ax_d.set_title("(d) Cosine từng câu, không qua projection", fontsize=8.5, color=INK,
               pad=9, loc="left")
ax_d.text(0.0, 1.015, f"{len(TEXTS):,} câu · PR {GAUGE_STATS['participation_ratio']:.2f}/{D_S}"
                      f" · top σ share {GAUGE_STATS['top_singular_share']:.2f}",
          transform=ax_d.transAxes, fontsize=7, color=MUTED, va="bottom")
ax_d.legend(loc="upper left", fontsize=6, frameon=False, handlelength=1.0,
            handletextpad=0.4, labelspacing=0.25)

figure.text(0.005, -0.03,
            f"$P_{{PCA}}$ giữ {ENERGY:.1%} năng lượng teacher · cosine trung bình "
            f"{GAUGE_STATS['cos_before']:+.3f} → {GAUGE_STATS['cos_after']:+.3f} · "
            f"{TEACHER_MODEL} → {STUDENT_MODEL}",
            fontsize=6.5, color=MUTED)

pdf_path = FIGURE_DIR / f"{FIGURE_STEM}.pdf"
figure.savefig(pdf_path, bbox_inches="tight")
figure.savefig(FIGURE_DIR / f"{FIGURE_STEM}.png", bbox_inches="tight")
print(f"Đã ghi {pdf_path}")
plt.show()

In [ ]:
# 8. Sidecar: mọi con số hình đang nói, để caption không phải chép tay
import json

sidecar = {
    "figure": FIGURE_STEM,
    "generated": RUN_STAMP,
    "commit": head,
    "pair": PAIR,
    "teacher_model": TEACHER_MODEL,
    "student_model": STUDENT_MODEL,
    "teacher_pooling": TEACHER_POOLING,
    "student_pooling": STUDENT_POOLING,
    "sentence_source": str(SOURCE_PATH),
    "sentences": len(TEXTS),
    "plotted": int(len(PLOT_INDEX)),
    "teacher_dim": D_T,
    "student_dim": D_S,
    "projection_rank": RANK,
    "explained_energy": float(ENERGY),
    "random_subspace_energy": RANK / D_T,
    "gauge": {key: float(value) for key, value in GAUGE_STATS.items()},
    "haar_seed": HAAR_SEED,
    "mean_cosine": {label: float(values.mean()) for label, values in COSINES.items()},
    "frame_centered": bool(FRAME_CENTER),
    "coordinate_span": SPAN,
    "reference_run": REFERENCE_RUN if IS_REFERENCE_CONFIG else None,
}
sidecar_path = FIGURE_DIR / f"{FIGURE_STEM}.json"
sidecar_path.write_text(json.dumps(sidecar, indent=2), encoding="utf-8")
print(f"Đã ghi {sidecar_path}")

print(f"""
\\caption{{Teacher interface của GATE-KD trên {len(TEXTS):,} câu.
(a) teacher $T$ ({D_T}D) và (b) student pretrained $Z_0$ ({D_S}D) sống trong hai
không gian khác nhau, mỗi panel nhìn qua hai hướng chính của chính nó.
(c) target sau $P_{{PCA}}$ (giữ {ENERGY:.1%} năng lượng) và sau khi thêm gauge, tất cả
nhìn qua đúng một khung $W_Z$ của panel (b).
(d) cosine từng câu giữa student và target: $R^\\star$ đưa trung bình từ
{GAUGE_STATS['cos_before']:+.3f} lên {GAUGE_STATS['cos_after']:+.3f}, trong khi một
phép xoay Haar cùng chi phí giữ nó ở {COSINES['PCA + Haar Q'].mean():+.3f}.
Cross-covariance có participation ratio {GAUGE_STATS['participation_ratio']:.2f} trên
{D_S} hướng, nên phép xoay gần như rank-one.
All alignments are computed in the original {D_S}-dimensional student space; plots show
fixed two-dimensional projections for visualization only. The figure shows $R$ at
initialization; the recipe refits it each epoch.}}
""".strip())